<a href="https://colab.research.google.com/github/VanaheimD/Cryptography/blob/main/lab_3a_asymmetric_cryptography.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from Crypto.PublicKey import RSA
from Crypto.Cipher import PKCS1_OAEP,AES
from Crypto.Random import random, get_random_bytes
from Crypto.Hash import SHA256
from Crypto.Util.Padding import pad, unpad

In [ ]:
def DH_Public_number():
  p = 23
  g = 5
  return p,g

def generate_keypairs(p,g):
  private_key = random.randint(1,10)
  public_key = pow(g,private_key)%p
  return private_key,public_key

def compute_secret(public_key,private_key,p):
  secret = pow(public_key,private_key)%p
  return secret

In [ ]:
from Crypto import PublicKey
def generate_keypairs_RSA():
  key = RSA.generate(2048)
  private_key = key.export_key()
  public_key = key.public_key().export_key()
  return private_key,public_key

def rsa_encrypt(message,public_key_byte):
  public_key = RSA.import_key(public_key_byte)
  cipher = PKCS1_OAEP.new(public_key)
  ciphertext = cipher.encrypt(message.encode())
  return ciphertext


def rsa_decrypt(ciphertext,private_key_byte):
  private_key = RSA.import_key(private_key_byte)
  cipher = PKCS1_OAEP.new(private_key)
  message = cipher.decrypt(ciphertext).decode()
  return message

In [ ]:
def derive_aes_key(shared_secret):
  secret_bytes = str(shared_secret).encode()
  key = SHA256.new(secret_bytes).digest()
  return key

def aes_encrypt(message,key):
  iv = get_random_bytes(16)
  cipher = AES.new(key,AES.MODE_CBC,iv)
  ciphertext = cipher.encrypt(pad(message.encode(),AES.block_size))
  return iv,ciphertext

def aes_decrypt(iv,ciphertext,key):
  cipher = AES.new(key,AES.MODE_CBC,iv)
  message = unpad(cipher.decrypt(ciphertext),AES.block_size).decode()
  return message


In [ ]:
if __name__ == "__main__":
  print("Task1:Diffie-Hellman Key Exchange")
  p,g = DH_Public_number()
  Alice_private_key,Alice_public_key = generate_keypairs(p,g)
  Bob_private_key,Bob_public_key = generate_keypairs(p,g)

  Alice_shared = compute_secret(Bob_public_key,Alice_private_key,p)
  Bob_shared = compute_secret(Alice_public_key,Bob_private_key,p)

  print("Alice public key:",Alice_public_key)
  print("Bob public key:",Bob_public_key)
  print("Alice shared secret:",Alice_shared)
  print("Bob shared secret:",Bob_shared)
  print("shared secret match:", Alice_shared == Bob_shared)

  print("\nTask2:RSA encrytion/decryption")
  Bob_private_key,Bob_public_key = generate_keypairs_RSA()
  message = "hello bob"
  ciphertext = rsa_encrypt(message,Bob_public_key)
  decrypt_message = rsa_decrypt(ciphertext,Bob_private_key)

  print("Original message:", message)
  print("Encrypted message:", ciphertext)
  print("Decrypted message:", decrypt_message)

  print("\nTask3:AES encryption/decryption")
  aes_key = derive_aes_key(Alice_shared)
  secret_message = "message protect with AES."
  iv,encrypt_message = aes_encrypt(secret_message,aes_key)
  decrypt_message = aes_decrypt(iv,encrypt_message,aes_key)

  print("Original AES message:", secret_message)
  print("IV(hex):", iv.hex)
  print("AES Encrypted message:", encrypt_message)
  print("AES Decrypted message:", decrypt_message)



Task1:Diffie-Hellman Key Exchange
Alice public key: 11
Bob public key: 11
Alice shared secret: 19
Bob shared secret: 19
shared secret match: True

Task2:RSA encrytion/decryption
Original message: hello bob
Encrypted message: b'\tI\xaeIT\xe7\x01*\x02\xe4k\xe1\xf3?\xd2xF\x8a\xda\x94\xafoK\xf2\x8d\xf9\xdf\xba\xe4\xa4\xc2^"M\xc5/\xc1q\x04\xcb\x01\xe5\xa1\t\xc4)\xd8\\H\xc4a"\xb0\xe4\xca\xd5\x12\x96d\xb6\x1a\xceFR\xcd\xc3\xe9$\xd8r\t\x19\x05\x03\x12^\x84\x87\x8dm5\x0c\x8cm\xad\xa3?\xf09\x03]Q=\xe2\xffkXy\xe8\xef^>:\x8b\xae&!\x8c\x1fD\xf0\xac\xd7<\xe5\t\x1f_Q\x13&\xd3\x9a\x18j\xdc\xdbZ\x9b)\xd3\x89\xd1-t\xcf\xf5U\xb6\x0fE*\xa8\x1f$\x8c\x1c\x19\xfa\xb2\xe2\x9d8\xf8\xa8tTG\x1d\x01k"\n#h\xa8\xef]X\xff\x99\x80\x1f\x85 \xe1\xec\x05\x9b\x04R\xb1M=\xf6\x8a\x14\xe6J\xb8-\xc9\xda8\x8e-\x13\x04\xd56\xb1PjA\xfb\x12\xe9\xa0G\xafK*\x1d\xe4^\xd0\xa9\x18\xe0\x8f\xe4\x88-\xb9\xba\x00\x95)vc!\x9a\xfdA\x91(\x8a\n\x0eb\xb6\x06\xc6\xf2W\xcbs\x82\xa5R4\xf6U\x10\xba\x0f'
Decrypted message: hello bob

Task3:AES enc